# Data Preparation - Norman Dataset

This notebook loads the Norman perturbation dataset, prepares train/test splits, and saves them for downstream analysis.

## 1. Import Libraries

In [5]:
import sys
import os
from pathlib import Path
import numpy as np
import pickle

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Force all operations to use D drive
os.environ['HOME'] = str(project_root)
os.environ['USERPROFILE'] = str(project_root)
os.environ['TEMP'] = str(project_root / 'temp')
os.environ['TMP'] = str(project_root / 'temp')

# Create temp directory on D drive
(project_root / 'temp').mkdir(exist_ok=True)

from gears import PertData, GEARS

## 2. Load Norman Dataset

In [6]:
# Initialize PertData
data_path = '../data'
pert_data = PertData(data_path)

# Load Norman dataset
print("Loading Norman dataset...")
pert_data.load(data_name='norman')
print("Dataset loaded successfully!")

Found local copy...
Found local copy...


Loading Norman dataset...


Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!


Dataset loaded successfully!


## 3. Prepare Data Split

Split options:
- `simulation`: Simulate single/combo perturbations
- `combo_seen0`: No combos seen during training
- `combo_seen1`: One gene in combo seen during training
- `combo_seen2`: Both genes in combo seen during training

In [7]:
# Ensure splits directory exists
splits_dir = Path(data_path) / 'norman' / 'splits' / 'data'
splits_dir.mkdir(parents=True, exist_ok=True)
print(f"Splits directory ready: {splits_dir}")

Splits directory ready: ..\data\norman\splits\data


In [8]:
# Prepare split
split_type = 'simulation'
random_seed = 1

print(f"Preparing data split (type={split_type}, seed={random_seed})...")
pert_data.prepare_split(split=split_type, seed=random_seed)
print("Data split prepared!")

Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!


Preparing data split (type=simulation, seed=1)...
here1
Data split prepared!


## 4. Create Dataloaders

In [9]:
# Create dataloaders
batch_size = 32
test_batch_size = 128

print(f"Creating dataloaders (batch_size={batch_size}, test_batch_size={test_batch_size})...")
pert_data.get_dataloader(batch_size=batch_size, test_batch_size=test_batch_size)
print("Dataloaders created!")

Creating dataloaders....

Creating dataloaders (batch_size=32, test_batch_size=128)...



Done!


Dataloaders created!


## 5. Inspect the Data

In [10]:
# Check if pert_data has the splits
print("Dataset Information:")
print(f"  Data path: {data_path}")
print(f"  Split type: {split_type}")
print(f"  Random seed: {random_seed}")
print(f"  Train batch size: {batch_size}")
print(f"  Test batch size: {test_batch_size}")

Dataset Information:
  Data path: ../data
  Split type: simulation
  Random seed: 1
  Train batch size: 32
  Test batch size: 128


## 6. Save Train and Test Sets

In [11]:
# The data is already saved by GEARS in the data/norman directory
# We don't need to pickle the entire pert_data object
# Instead, just save the configuration for reproducibility

processed_dir = Path(data_path) / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

print("Data is already stored in the data/norman directory by GEARS")
print(f"Processed directory created at: {processed_dir}")
print("\nTo reload this data later, simply run:")
print("  pert_data = PertData('./data')")
print("  pert_data.load(data_name='norman')")
print("  pert_data.prepare_split(split='simulation', seed=1)")
print("  pert_data.get_dataloader(batch_size=32, test_batch_size=128)")

Data is already stored in the data/norman directory by GEARS
Processed directory created at: ..\data\processed

To reload this data later, simply run:
  pert_data = PertData('./data')
  pert_data.load(data_name='norman')
  pert_data.prepare_split(split='simulation', seed=1)
  pert_data.get_dataloader(batch_size=32, test_batch_size=128)


## 7. Save Configuration Info

In [13]:
# Save configuration as a text file
config_path = processed_dir / 'norman_config.txt'

config_info = f"""Norman Dataset Configuration
==============================
Dataset: norman
Split Type: {split_type}
Random Seed: {random_seed}
Train Batch Size: {batch_size}
Test Batch Size: {test_batch_size}
Data Path: {data_path}
Splits Directory: {splits_dir}
"""

with open(config_path, 'w') as f:
    f.write(config_info)

print(config_info)
print(f"Configuration saved to {config_path}")

Norman Dataset Configuration
Dataset: norman
Split Type: simulation
Random Seed: 1
Train Batch Size: 32
Test Batch Size: 128
Data Path: ../data
Splits Directory: ..\data\norman\splits\data

Configuration saved to ..\data\processed\norman_config.txt


## 8. Quick Data Inspection

In [14]:
# Display basic statistics if pert_data has accessible attributes
print("\nData structure:")
print(f"PertData object: {type(pert_data)}")
print(f"Available attributes: {[attr for attr in dir(pert_data) if not attr.startswith('_')][:10]}...")


Data structure:
PertData object: <class 'gears.pertdata.PertData'>
Available attributes: ['adata', 'create_cell_graph', 'create_cell_graph_dataset', 'create_dataset_file', 'ctrl_adata', 'data_path', 'dataloader', 'dataset_name', 'dataset_path', 'dataset_processed']...


## 9. SVM

In [ ]:
# Extract data from GEARS AnnData for SVM (like in pertubation_data_analysis_exercise)
import sys
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Ensure src is in path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.svm_model import SVMModel

# Get the AnnData object (contains raw gene expression like pertdata from exercise)
adata = pert_data.adata

# Extract gene expression matrix X
X = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X  # Convert sparse to dense if needed
print(f"Gene expression matrix: {X.shape}")

# Extract perturbation labels
y_labels = adata.obs['condition'].values  # Perturbation conditions
print(f"Labels shape: {y_labels.shape}")
print(f"Unique perturbations: {len(set(y_labels))}")
print(f"Sample labels: {y_labels[:5]}")

# Encode string labels to integers
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_labels)
print(f"\nEncoded labels range: {y.min()} to {y.max()}")

# Split into train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_seed, stratify=y
)

print(f"\nTraining data: X_train.shape={X_train.shape}, y_train.shape={y_train.shape}")
print(f"Test data: X_test.shape={X_test.shape}, y_test.shape={y_test.shape}")

# Create and train SVM
print("\n" + "="*50)
print("Training SVM...")
print("="*50)
model = SVMModel(kernel='linear', C=1.0, pca_components=100)
model.fit(X_train, y_train)

# Evaluate on test set
results = model.evaluate(X_test, y_test)
print("\nTest Results:")
for metric, value in results.items():
    print(f"  {metric}: {value:.4f}")


Gene expression matrix: (89357, 5045)
Labels shape: (89357,)
Unique perturbations: 277
Sample labels: ['TSC22D1+ctrl', 'KLF1+MAP2K6', 'ctrl', 'CEBPE+RUNX1T1', 'MAML2+ctrl']
Categories (277, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']

Encoded labels range: 0 to 276

Training data: X_train.shape=(71485, 5045), y_train.shape=(71485,)
Test data: X_test.shape=(17872, 5045), y_test.shape=(17872,)

Training SVM...
